In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image

from ml_security import ChallengeClient
from ml_security.utils import save_image

In [ ]:
client = ChallengeClient.from_env()

run_dir    = Path("notebook_runs/challenge4_backdoor")
run_dir.mkdir(parents=True, exist_ok=True)

info_path        = run_dir / "task4_info.json"
model_path       = run_dir / "backdoored_model.pt"
trigger_mask_path = run_dir / "trigger_mask.png"

print(f"Run directory  : {run_dir}")
print(f"Info cache     : {info_path}")
print(f"Model weights  : {model_path}")
print(f"Trigger mask   : {trigger_mask_path}")

In [ ]:
info = client.get_task4_info()
info_path.write_text(json.dumps(info, indent=2))

print("=== Task description ===")
print(info.get("task_description", ""))
print()
print(f"Model architecture : {info['model_architecture']}")
print(f"Input shape        : {info['input_shape']}")
print(f"Num classes        : {info['num_classes']}")
print(f"Dataset            : {info['dataset']}")

In [ ]:
if model_path.exists():
    print(f"[skip] Model already on disk: {model_path}")
else:
    client.download_model(save_path=str(model_path))
    print(f"[download] Saved model to {model_path}")

print(f"File size: {model_path.stat().st_size / 1024:.1f} KB")

In [ ]:
state_dict = torch.load(model_path, map_location="cpu", weights_only=True)

print("State dict keys and shapes:")
for key, tensor in state_dict.items():
    print(f"  {key:<35} {str(tuple(tensor.shape)):<20} dtype={tensor.dtype}")

print()
total_params = sum(t.numel() for t in state_dict.values())
print(f"Total parameters: {total_params:,}")

In [ ]:
# Reconstruct the FCNet described in task4/info so we can do forward passes
class FCNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x.view(x.size(0), -1))


model = FCNet()
model.load_state_dict(state_dict)
model.eval()
print(model)
print()
print("Weights loaded successfully")

In [ ]:
# Probe the model with zero / random / constant inputs to get a feel for its baseline behaviour
rng = np.random.default_rng(42)

probes = {
    "all-zeros"  : np.zeros((1, 1, 28, 28), dtype=np.float32),
    "all-ones"   : np.ones ((1, 1, 28, 28), dtype=np.float32),
    "uniform"    : rng.uniform(0, 1, (1, 1, 28, 28)).astype(np.float32),
    "half-gray"  : np.full ((1, 1, 28, 28), 0.5, dtype=np.float32),
}

print(f"{'probe':<15} {'predicted class':>15}  {'top-3 probs'}")
print("-" * 55)
for name, arr in probes.items():
    with torch.no_grad():
        logits = model(torch.from_numpy(arr))
        probs  = torch.softmax(logits, dim=1)[0].numpy()
    top3 = np.argsort(probs)[::-1][:3]
    top3_str = ", ".join(f"{c}:{probs[c]:.3f}" for c in top3)
    print(f"{name:<15} {int(np.argmax(probs)):>15}  {top3_str}")

In [ ]:
# Look at per-neuron weight norms in the first layer — anomalous neurons
# can indicate which input pixels are over-weighted by the backdoor trigger
first_weight = state_dict[[k for k in state_dict if "weight" in k][0]]  # (512, 784)
pixel_norms  = first_weight.abs().sum(dim=0).numpy()  # (784,)
pixel_map    = pixel_norms.reshape(28, 28)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(pixel_map, cmap="hot")
axes[0].set_title("L1 weight norm per input pixel (layer 1)")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0])

axes[1].hist(pixel_norms, bins=50, edgecolor="white", linewidth=0.4)
axes[1].set_title("Distribution of pixel weight norms")
axes[1].set_xlabel("L1 norm")
axes[1].set_ylabel("count")

plt.suptitle("First-layer weight analysis", fontsize=12)
plt.tight_layout()
plt.show()

# Print the top-20 most influential input pixels
top_pixels = np.argsort(pixel_norms)[::-1][:20]
print("Top-20 pixels by L1 norm:")
for rank, idx in enumerate(top_pixels, 1):
    row, col = divmod(int(idx), 28)
    print(f"  #{rank:<3} pixel {idx:>4}  (row={row:2d}, col={col:2d})  norm={pixel_norms[idx]:.4f}")